# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Display dataset title and description
meta = dataset.metadata
# Access to_json() method for metadata
metadata_dict = meta.to_json()
print('Dataset title:', metadata_dict['name'])
print('Dataset description:', metadata_dict['description'])

## 2. Data Overview
Review available record sets, fields, and their IDs.

Every entity (record set, field, column) is referenced by its `@id` for consistency.

In [ ]:
# Print all record sets and fields with their @id
croissant_json = dataset.metadata.to_json()

# Helper: Find all record sets
record_sets = croissant_json.get('recordSet', [])
if not record_sets:
    print('No record sets found in metadata, loading as flat tabular record set.')
    # As Croissant datasets sometimes have their record sets specified as a distribution
    record_sets = [dist['@id'] for dist in croissant_json.get('distribution', [])]

record_set_ids = []
print('Record Sets found:')
for rs in record_sets:
    if isinstance(rs, dict):
        rs_id = rs.get('@id', None)
        if rs_id:
            record_set_ids.append(rs_id)
            print(f"- @id: {rs_id}")
    elif isinstance(rs, str):
        record_set_ids.append(rs)
        print(f"- @id: {rs}")

# Fetch and print fields for each record set
print('\nFields within each record set:')
for rs_id in record_set_ids:
    try:
        info = dataset.record_set(rs_id)
        fields = info.fields
        print(f"Record Set @id: {rs_id}")
        for f in fields:
            print(f"  - Field @id: {f['@id']}, name: {f['name']} (type: {f.get('dataType', '')})")
    except Exception as e:
        print(f"Could not fetch fields for record set {rs_id}: {e}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.
Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract all records from each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame for record set @id: {record_set_id}")
        print("Columns:", df.columns.tolist())
        print("Head:")
        display(df.head())
    else:
        print(f"No records found for record set @id: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—filtering records, normalizing numeric fields, and grouping data.

All field references use their `@id`. Please adjust the numeric and group field IDs as appropriate for your dataset.

In [ ]:
# Example: Select a numeric field (e.g., age) to filter and normalize
# Inspect available fields to pick the correct field @id

# Choose first tabular record set for demo
if dataframes:
    selected_rs_id = list(dataframes)[0]
    df = dataframes[selected_rs_id]
    print(f'Selected record set for EDA: {selected_rs_id}')

    # Find potential numeric fields (e.g., age, interval, etc.)
    numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or df[col].dtype in ['int64', 'float64']]
    print('Detected numeric fields:', numeric_fields)

    if numeric_fields:
        numeric_field = numeric_fields[0]  # Choose one for analysis
        print(f'Using {numeric_field} for numeric analysis')

        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f'Filtered records where {numeric_field} > {threshold}:')
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f'Normalized {numeric_field} for filtered records:')
        display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

        # Group by a categorical/grouping field (e.g., anatomical location, sex)
        group_field_candidates = [col for col in df.columns if 'location' in col.lower() or 'sex' in col.lower() or 'msi' in col.lower() or df[col].dtype == 'object']
        print('Detected group fields:', group_field_candidates)
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f'Grouping by field: {group_field}')
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f'Grouped summary by {group_field}:')
            display(grouped_df.head())
    else:
        print('No suitable numeric field found for filtering and normalization.')
else:
    print('No tabular record sets loaded. Please check record set IDs and dataset structure.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
# Example: Visualize distribution of selected numeric field and relationship to group field
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    rs_id = list(dataframes)[0]
    df = dataframes[rs_id]
    # Try plotting if EDA picked field names
    numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or df[col].dtype in ['int64', 'float64']]
    group_field_candidates = [col for col in df.columns if 'location' in col.lower() or 'sex' in col.lower() or 'msi' in col.lower() or df[col].dtype == 'object']

    if numeric_fields:
        numeric_field = numeric_fields[0]
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_field], bins=15, kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.show()

        if group_field_candidates:
            group_field = group_field_candidates[0]
            plt.figure(figsize=(7,4))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f'{numeric_field} by {group_field}')
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.show()
    else:
        print('No numeric field available for visualization.')
else:
    print('No record set loaded for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides rich clinical and pathological information for second primary colorectal cancer cases in survivors.
- Using the `mlcroissant` library, we inspected available record sets, fields via their `@id`s, and loaded them as tabular DataFrames.
- Exploratory analyses and visualizations revealed distributions and relationships of numeric (e.g., age) and categorical (e.g., anatomical location) fields.
- Further domain-specific investigations can be performed using the grouped and filtered results prepared here.